# Phase 2 — Demand regression (first-differenced, Newey–West)

End-to-end demand-model fitting on `master_monthly.csv`:

1. ADF unit-root tests on arrivals and three real-FX series in **levels**.
2. First-differencing and a second ADF round on the differenced series.
3. OLS of $\Delta \text{arrivals}_t$ on $\Delta \text{real EUR/TRY}_t$, $\Delta \text{real GBP/TRY}_t$, lag-1 DE & GB Trends, and the three shock flags. Standard errors are **Newey–West (HAC, 4 lags)** to keep inference robust to monthly autocorrelation that survives differencing.
4. Coefficient interpretation, with economic reading of the real-FX channel.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.stattools import durbin_watson
from pathlib import Path

ROOT = Path('..').resolve()
df = pd.read_csv(ROOT / 'data' / 'processed' / 'master_monthly.csv',
                 index_col=0, parse_dates=True)
df.index.name = 'date'
print(df.shape)

(190, 25)


## 1. ADF — levels

Augmented Dickey–Fuller with constant (no trend) on each level series. The null hypothesis is **unit root** (non-stationary); reject if $p < 0.05$. `maxlag` is chosen by AIC.

In [2]:
def adf_row(name: str, s: pd.Series) -> dict:
    s = s.dropna()
    stat, pval, used_lag, nobs, crit, _icbest = adfuller(
        s, regression='c', autolag='AIC')
    return {
        'series':     name,
        'n':          int(nobs),
        'used_lag':   int(used_lag),
        'adf_stat':   round(float(stat), 3),
        'p_value':    round(float(pval), 4),
        'crit_5pct':  round(float(crit['5%']), 3),
        'conclusion': 'stationary' if pval < 0.05 else 'non-stationary',
    }

LEVEL_SERIES = ['arrivals_total', 'real_EUR_TRY', 'real_GBP_TRY', 'real_USD_TRY']
adf_levels = pd.DataFrame([adf_row(n, df[n]) for n in LEVEL_SERIES])
adf_levels

,series,n,used_lag,adf_stat,p_value,crit_5pct,conclusion
0,arrivals_total,104,13,-1.805,0.3781,-2.890,non-stationary
1,real_EUR_TRY,178,4,-1.146,0.6963,-2.878,non-stationary
2,real_GBP_TRY,172,10,-1.166,0.6882,-2.878,non-stationary
3,real_USD_TRY,184,4,-1.014,0.7482,-2.877,non-stationary


## 2. First-difference, re-test

Series flagged as non-stationary in step 1 are first-differenced ($\Delta x_t = x_t - x_{t-1}$) and the ADF is re-run. The differenced series feed the regression in step 3.

In [3]:
non_stat = adf_levels.loc[adf_levels['conclusion'] == 'non-stationary', 'series'].tolist()
print('Non-stationary in levels:', non_stat)

for col in non_stat:
    df[f'd_{col}'] = df[col].diff()

adf_diff = pd.DataFrame([adf_row(f'd_{c}', df[f'd_{c}']) for c in non_stat])
adf_diff

Non-stationary in levels: ['arrivals_total', 'real_EUR_TRY', 'real_GBP_TRY', 'real_USD_TRY']


,series,n,used_lag,adf_stat,p_value,crit_5pct,conclusion
0,d_arrivals_total,104,12,-2.782,0.0609,-2.890,non-stationary
1,d_real_EUR_TRY,178,3,-8.918,0.0000,-2.878,stationary
2,d_real_GBP_TRY,172,9,-4.918,0.0000,-2.878,stationary
3,d_real_USD_TRY,184,3,-9.091,0.0000,-2.877,stationary


## 3. OLS regression with Newey–West SE

**Specification**

$$\Delta \text{arrivals}_t = \alpha
  + \beta_1 \Delta \text{rEUR/TRY}_t
  + \beta_2 \Delta \text{rGBP/TRY}_t
  + \beta_3 \text{Trends}^{DE}_{t-1}
  + \beta_4 \text{Trends}^{GB}_{t-1}
  + \gamma_1 \text{covid}_t + \gamma_2 \text{war}_t + \gamma_3 \text{mideast}_t
  + \varepsilon_t$$

Trends regressors enter at **lag 1** (so they reflect prior-month search intent, consistent with the CCF peaks). Standard errors are HAC with `maxlags=4` — captures up to a quarter of monthly autocorrelation.

_Bandwidth note._ This notebook uses $\Delta_1$ (first differences), so a quarterly HAC bandwidth is appropriate. Notebooks 04 and 05 use $\Delta_{12}$ (YoY) instead: overlapping 12-month differences mechanically induce an MA(11) error structure, so those notebooks widen HAC to `maxlags=12`.

In [4]:
model_df = df.assign(
    trends_DE_lag1 = df['trends_DE_Türkei_Urlaub'].shift(1),
    trends_GB_lag1 = df['trends_GB_Turkey_holiday'].shift(1),
)

regressors = [
    'd_real_EUR_TRY', 'd_real_GBP_TRY',
    'trends_DE_lag1', 'trends_GB_lag1',
    'covid', 'russia_war', 'mideast',
]
y_col = 'd_arrivals_total'

sample = model_df[[y_col] + regressors].dropna()
y = sample[y_col]
X = sm.add_constant(sample[regressors])
print(f'Sample: {sample.index.min().date()} → {sample.index.max().date()}  (N={len(sample)})')

model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
print(model.summary())
print()
print(f'Durbin–Watson on residuals: {durbin_watson(model.resid):.3f}')

Sample: 2016-02-01 → 2025-03-01  (N=110)
                            OLS Regression Results                            
Dep. Variable:       d_arrivals_total   R-squared:                       0.089
Model:                            OLS   Adj. R-squared:                  0.027
Method:                 Least Squares   F-statistic:                     3.140
Date:                Sun, 19 Jul 2026   Prob (F-statistic):            0.00480
Time:                        23:10:12   Log-Likelihood:                -1670.1
No. Observations:                 110   AIC:                             3356.
Df Residuals:                     102   BIC:                             3378.
Df Model:                           7                                         
Covariance Type:                  HAC                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
con

## 4. Interpretation

The cell below derives the prose conclusions from the fitted model — sign / significance / economic reading are computed from `model.params` and `model.pvalues`, not hand-written, so the text stays accurate if the underlying CSV is refreshed.

In [5]:
alpha = 0.05
params = model.params
pvals = model.pvalues

print('Significant regressors (p < 0.05):')
sig = pvals[pvals < alpha]
if sig.empty:
    print('  (none)')
else:
    for name in sig.index:
        print(f'  {name:18s}  beta={params[name]:+.3e}  p={pvals[name]:.4f}')

print()
print('Real EUR/TRY channel — economic reading:')
b = params.get('d_real_EUR_TRY', np.nan)
p = pvals.get('d_real_EUR_TRY', np.nan)
if np.isnan(b):
    print('  (regressor not in model)')
elif p >= alpha:
    print(f'  Coefficient {b:+.0f} is NOT significant at 5% (p={p:.3f}).')
    print('  The data do not support a same-month real-FX → arrivals channel for the Eurozone in this spec.')
elif b > 0:
    print(f'  Coefficient {b:+.0f} (p={p:.3f}): a one-unit rise in real EUR/TRY')
    print('  (TRY one index point cheaper vs. EUR in purchasing-power terms) is associated')
    print(f'  with ~{b:+.0f} additional arrivals in the same month — the expected substitution channel.')
else:
    print(f'  Coefficient {b:+.0f} (p={p:.3f}) is significant but NEGATIVE — opposite of the textbook prior.')
    print('  Likely interpretations: monthly differencing introduces noise that swamps the FX signal,')
    print('  or the differenced regressor is picking up reverse-causal short-run dynamics.')

print()
print(f'R-squared: {model.rsquared:.3f}   Adj R²: {model.rsquared_adj:.3f}')
dw = durbin_watson(model.resid)
print(f'Durbin–Watson: {dw:.3f}  '
      f'(≈2 = no autocorr; <1.5 = positive autocorr; >2.5 = negative autocorr)')
if dw < 1.5:
    print('  → residuals show positive autocorrelation; the Newey–West SE already accounts for this,')
    print('    but a richer dynamic spec (AR term or lagged dependent) would tighten inference.')

Significant regressors (p < 0.05):
  const               beta=-4.938e+05  p=0.0124

Real EUR/TRY channel — economic reading:
  Coefficient +59301 is NOT significant at 5% (p=0.212).
  The data do not support a same-month real-FX → arrivals channel for the Eurozone in this spec.

R-squared: 0.089   Adj R²: 0.027
Durbin–Watson: 1.299  (≈2 = no autocorr; <1.5 = positive autocorr; >2.5 = negative autocorr)
  → residuals show positive autocorrelation; the Newey–West SE already accounts for this,
    but a richer dynamic spec (AR term or lagged dependent) would tighten inference.
